# Notebook 02 — Klasifikasi: Prediksi Kategori Nilai Transaksi

**Fase 2 · Minilab EduBI · Data Mining**

---

## Tujuan
Membangun model klasifikasi untuk memprediksi **kategori nilai transaksi** (`revenue_category`):
- `High Value`  — transaksi ≥ Rp 10.000.000
- `Mid Value`   — transaksi Rp 3.000.000 – Rp 9.999.999
- `Low Value`   — transaksi < Rp 3.000.000

Model dibangun dengan **Random Forest** dan hasilnya dicatat ke **MLflow**.

> **Catatan dataset**: Data sample 50 transaksi hanya memiliki status `done`.
> Target `revenue_category` dipilih karena memiliki variasi kelas yang cukup untuk klasifikasi.

---
## 1. Setup & Koneksi

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import clickhouse_connect
import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

CH_HOST  = os.getenv('CH_HOST', 'localhost')
CH_PORT  = int(os.getenv('CH_PORT', 8123))
CH_USER  = os.getenv('CH_USER', 'default')
CH_PASS  = os.getenv('CH_PASSWORD', '')
MLFLOW_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://localhost:5000')

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('02_classification_order_status')

client = clickhouse_connect.get_client(
    host=CH_HOST, port=CH_PORT,
    username=CH_USER, password=CH_PASS
)
print('Koneksi ClickHouse berhasil.')

---
## 2. Load Data dari Silver Layer


In [ ]:
query = """
SELECT
    order_id,
    customer_id,
    category,
    quantity,
    toFloat64(unit_price)   AS unit_price,
    toFloat64(total_price)  AS total_price,
    order_year,
    order_month,
    branch,
    revenue_category
FROM silver.silver_sales
WHERE status = 'done'
"""

df = client.query_df(query)
print(f'Total records: {len(df)}')
print('\nDistribusi target (revenue_category):')
print(df['revenue_category'].value_counts())
df.head()

---
## 3. Feature Engineering


In [ ]:
df_model = df.copy()

# Encode target (multi-class)
le_target = LabelEncoder()
df_model['target'] = le_target.fit_transform(df_model['revenue_category'])
class_names = le_target.classes_
print('Label encoding target:', dict(zip(class_names, le_target.transform(class_names))))

# Categorical encoding fitur
le = LabelEncoder()
for col in ['category', 'branch']:
    df_model[col + '_enc'] = le.fit_transform(df_model[col].astype(str))

# Fitur final
# CATATAN: total_price & unit_price sengaja DIHAPUS — keduanya secara langsung
# mendefinisikan revenue_category sehingga menyebabkan data leakage (akurasi 1.0 palsu).
# Model yang baik belajar dari konteks transaksi, bukan dari nilai yang menjadi target.
FEATURES = [
    'quantity',
    'order_year', 'order_month',
    'category_enc', 'branch_enc'
]
TARGET = 'target'

X = df_model[FEATURES]
y = df_model[TARGET]

print('\nFitur (tanpa total_price/unit_price untuk menghindari data leakage):')
print(FEATURES)
print('Shape X:', X.shape)
print('\nDistribusi kelas:')
for label, count in zip(class_names, [sum(y==i) for i in range(len(class_names))]):
    print(f'  {label}: {count}')

---
## 4. Train/Test Split


In [ ]:
# Gunakan stratify hanya jika semua kelas punya minimal 2 sampel
min_class_count = y.value_counts().min()
stratify_param = y if min_class_count >= 2 else None
if stratify_param is None:
    print('⚠ Stratify dinonaktifkan: ada kelas dengan < 2 sampel')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=stratify_param
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print('Kelas di y_train:', sorted(y_train.unique()))
print('Kelas di y_test :', sorted(y_test.unique()))

---
## 5. Training & Evaluasi dengan MLflow


In [ ]:
PARAMS = {
    'n_estimators':     100,
    'max_depth':        5,
    'min_samples_leaf': 2,
    'class_weight':     'balanced',
    'random_state':     42
}

mlflow.set_experiment('02_classification_revenue_category')

with mlflow.start_run(run_name='random_forest_v1'):
    model = RandomForestClassifier(**PARAMS)
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    n_classes = len(np.unique(y_test))
    metrics = {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall':    recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1':        f1_score(y_test, y_pred, average='weighted', zero_division=0),
    }

    # ROC-AUC multi-class (hanya jika semua kelas ada di test set)
    if n_classes == len(model.classes_) and n_classes > 1:
        metrics['roc_auc_ovr'] = roc_auc_score(
            y_test, y_proba, multi_class='ovr', average='weighted'
        )
    else:
        metrics['roc_auc_ovr'] = float('nan')
        print('⚠ ROC-AUC dilewati: tidak semua kelas ada di test set')

    mlflow.log_params(PARAMS)
    mlflow.log_metrics({k: v for k, v in metrics.items() if v == v})  # skip NaN
    mlflow.sklearn.log_model(model, 'random_forest_model')

    print('\nMetrik evaluasi:')
    for k, v in metrics.items():
        print(f'  {k:<14}: {v:.4f}' if v == v else f'  {k:<14}: N/A')

print('\n✅ Run MLflow selesai')

In [ ]:
# Classification Report
print(classification_report(y_test, y_pred, target_names=class_names))

---
## 6. Confusion Matrix & Feature Importance

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=class_names,
    cmap='Blues', ax=ax1
)
ax1.set_title('Confusion Matrix — Revenue Category')

# Feature Importance
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
importances.plot.barh(ax=ax2, color='steelblue')
ax2.set_title('Feature Importance')
ax2.set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('experiments/classification_results.png', dpi=100)
plt.show()

---
## 7. Kesimpulan

**Pertanyaan Diskusi:**
1. Fitur apa yang paling berpengaruh dalam memprediksi kategori nilai transaksi?
2. Mengapa `class_weight='balanced'` penting untuk dataset yang tidak seimbang?
3. Apa perbedaan antara `precision` dan `recall` dalam konteks bisnis ini?
4. Bagaimana cara meningkatkan performa model jika data lebih banyak tersedia?
5. Apakah fitur `total_price` sebaiknya digunakan sebagai fitur? Mengapa?

**Lihat hasil eksperimen di MLflow:** http://localhost:5000